# Stage C — collect rows into the ablation table

Upload the `results_*.json` files from every machine into one folder (or attach
the Stage B outputs as inputs), then run. CPU only.


In [ ]:
import os, glob, json
files = sorted(glob.glob("/kaggle/input/**/results_*.json", recursive=True) +
               glob.glob("./results_*.json") + glob.glob("/content/**/results_*.json", recursive=True))
R = {json.load(open(f))["run"]: json.load(open(f)) for f in files}
print("found:", list(R))

In [ ]:
ORDER = ["baseline", "rope", "rope_gqa", "rope_gqa_matched"]
hdr = f"{'run':<20}{'params':>9}{'kv':>4}{'bpb':>8}{'cache@2048':>12}{'dec@256':>10}{'dec@2048':>10}"
print(hdr); print("-"*len(hdr))
for k in ORDER:
    if k not in R: continue
    d = R[k]; c2, c0 = d["ctx"]["2048"], d["ctx"]["256"]
    f = lambda c: f"{c['decode_tok_s']:.0f}" if "decode_tok_s" in c else "N/A"
    print(f"{k:<20}{d['params']/1e6:8.2f}M{d['n_kv_heads']:>4}{d['best_bpb']:8.4f}"
          f"{c2['kv_cache_MB']:11.2f}M{f(c0):>10}{f(c2):>10}")

if "rope" in R and "rope_gqa" in R:
    a, b = R["rope"], R["rope_gqa"]
    print(f"\nGQA vs MHA: cache {a['ctx']['2048']['kv_cache_MB']/b['ctx']['2048']['kv_cache_MB']:.2f}x smaller, "
          f"bpb {b['best_bpb']-a['best_bpb']:+.4f}")
    for ctx in ("256","1024","2048"):
        if "decode_tok_s" in a["ctx"][ctx] and "decode_tok_s" in b["ctx"][ctx]:
            print(f"  decode @{ctx:>4}: {b['ctx'][ctx]['decode_tok_s']/a['ctx'][ctx]['decode_tok_s']:.2f}x")
if "baseline" in R and "rope" in R:
    print(f"\nRoPE vs learned: bpb {R['rope']['best_bpb']-R['baseline']['best_bpb']:+.4f}, "
          f"params {(R['rope']['params']-R['baseline']['params'])/1e3:+.0f}K, "
          f"baseline @2048: {R['baseline']['ctx']['2048'].get('error','ran')[:60]}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k in ORDER:
    if k not in R or not R[k].get("log"): continue
    lg = R[k]["log"]
    ax[0].plot([p["iter"] for p in lg], [p["val_bpb"] for p in lg], label=k)
    ax[1].plot([p["elapsed"]/60 for p in lg], [p["val_bpb"] for p in lg], label=k)
ax[0].set_xlabel("iteration"); ax[1].set_xlabel("wall-clock (min)")
for a_ in ax: a_.set_ylabel("val bits-per-byte"); a_.legend(); a_.grid(alpha=.3)
plt.tight_layout(); plt.savefig("ablation.png", dpi=140); plt.show()